# 🎬 AI Factory Video Generator (Colab)

Generates **realistic factory B-roll** (Flow/Veo style) for the Lean Manufacturing course, using the **free, open-source LTX-Video** model on a Colab GPU.

**Style:** bright & corporate, modern **electronics** factory, diverse **Indian** workers (many women) in **ESD coats, gloves & masks** — one shot per waste, varied camera work.

**How to use**
1. Runtime → Change runtime type → **GPU** (T4 is fine; L4/A100 is faster & higher quality).
2. Run each cell top to bottom.
3. Clips are saved to `clips/` and zipped at the end → download and send to Claude to assemble.

> AI video can't spell — ignore any text it draws; Claude overlays the correct titles/labels afterwards.

In [ ]:
# 1) Check the GPU you were given
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Install dependencies (~2-3 min)
# After this cell finishes, do: Runtime -> Restart session, then run Cell 3.
!pip -q install "diffusers>=0.32.0" transformers accelerate sentencepiece imageio imageio-ffmpeg -U
!pip -q install -U "torchao>=0.9.0"
print("\n✅ Installed. NOW DO: Runtime -> Restart session, then run Cell 3 onward.")

In [ ]:
# 3) Load the LTX-Video model (text-to-video)
import torch
from diffusers import LTXPipeline

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video", torch_dtype=torch.bfloat16
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()  # saves VRAM on T4
print("Model ready.")

In [ ]:
# 4) The prompt pack — one cinematic shot per waste, consistent look.
# STYLE is prepended to every shot so all clips match. Each shot ends with its
# own CAMERA direction (close-up / sweeping / static / slow) for variety.

STYLE = (
    "Bright, clean, well-lit modern electronics manufacturing factory in India, "
    "diverse Indian workers including many women, wearing white ESD anti-static "
    "coats, gloves and face masks, crisp professional corporate cinematography, "
    "soft even lighting, shallow depth of field, 4k, highly detailed. "
)

NEGATIVE = (
    "worst quality, blurry, distorted, deformed hands, extra fingers, "
    "watermark, text, captions, subtitles, jittery, low resolution, dark, gloomy"
)

SHOTS = {
  # intro / establishing
  "01_intro_establish": "Establishing view down rows of tidy electronics assembly "
      "workstations, many workers busy, spacious bright facility. "
      "CAMERA: slow sweeping crane dolly gliding forward.",
  "02_assembly_robotarm": "A robotic arm precisely places a tiny component onto a green "
      "circuit board while a gloved female operator guides the line. "
      "CAMERA: extreme macro close-up, static locked-off, soft bokeh.",

  # D - Defects (phone PCB)
  "03_defects": "A female quality inspector examines a smartphone motherboard PCB under "
      "a magnifier lamp, spots a fault, and drops it into a bright red reject tray. "
      "CAMERA: close-up, slow push-in.",
  # O - Overproduction
  "04_overproduction": "Tall shelves overflowing with stacked finished-product boxes in "
      "the storage area, clearly far too much stock. "
      "CAMERA: slow sweeping pan across the shelves.",
  # W - Waiting
  "05_waiting": "A woman worker stands idle beside a paused machine, arms crossed, "
      "waiting, glancing at it, others blurred behind. "
      "CAMERA: medium static shot, very slight slow drift.",
  # N - Non-utilized talent
  "06_talent": "A thoughtful female engineer pauses with a bright idea, looking up, "
      "while a manager walks past in the background without noticing. "
      "CAMERA: close-up on her face, shallow focus, slow.",
  # T - Transportation
  "07_transportation": "A worker pushes a trolley loaded with parts across a long factory "
      "floor, travelling a large distance. "
      "CAMERA: smooth tracking shot following the trolley.",
  # I - Inventory
  "08_inventory": "Rows of plastic bins and trays packed with work-in-progress parts "
      "piled high beside an assembly line. "
      "CAMERA: slow push-in, static framing.",
  # M - Motion
  "09_motion": "An operator at a cluttered bench reaches, stretches and bends repeatedly "
      "searching for a tool, wasted movement. "
      "CAMERA: tight close-up on the hands and arms, handheld-steady.",
  # E - Extra processing
  "10_extra_processing": "Gloved hands over-polishing and re-wiping an already-clean metal "
      "part far longer than needed, unnecessary effort. "
      "CAMERA: macro close-up on the hands, static.",

  # 5S tool board (replaces the kaizen shot)
  "11_5s_toolboard": "A pristine organized 5S shadow-board of hand tools, each tool in its "
      "outlined place, spotless and labelled, a worker returning one tool to its slot. "
      "CAMERA: slow pan across the board, close-up.",
  # outro
  "12_outro_efficient": "A bright, organized, efficient factory floor with neat workstations "
      "and busy workers, inspiring optimistic mood. "
      "CAMERA: sweeping rising crane shot.",
}
SHOTS = {k: STYLE + v for k, v in SHOTS.items()}
print(f"{len(SHOTS)} shots ready.")

In [ ]:
# 5) Generate every shot and save to clips/
import os
from diffusers.utils import export_to_video

os.makedirs("clips", exist_ok=True)

# 121 frames @ 24fps ~= 5s. Lower 'num_frames' if you run out of memory.
GEN = dict(width=768, height=512, num_frames=121,
           num_inference_steps=50, guidance_scale=3.0)

for name, prompt in SHOTS.items():
    out = f"clips/{name}.mp4"
    if os.path.exists(out):
        print("skip (exists):", name); continue
    print("generating:", name)
    video = pipe(prompt=prompt, negative_prompt=NEGATIVE, **GEN).frames[0]
    export_to_video(video, out, fps=24)
    print("  saved", out)
print("\nAll done.")

In [ ]:
# 6) (Optional) Re-generate ONE shot (e.g. if you didn't like it).
# Change the key, then run. It overwrites that clip.
from diffusers.utils import export_to_video
key = "03_defects"
v = pipe(prompt=SHOTS[key], negative_prompt=NEGATIVE, width=768, height=512,
         num_frames=121, num_inference_steps=50, guidance_scale=3.0).frames[0]
export_to_video(v, f"clips/{key}.mp4", fps=24)
print("re-saved", key)

In [ ]:
# 7) Zip all clips and download → send to Claude to assemble into the lesson
!zip -r clips.zip clips >/dev/null && echo 'zipped clips.zip'
from google.colab import files
files.download('clips.zip')